# Sanity 04 — the embeddings

Mirrors `scripts/04_extract_embeddings.py`. One plain forward pass per eval window; the artifact
is a parquet with ONE `embedding` column = the last position's 512-d hidden state, plus the
passthrough join keys. Sanity here: collapse check, same-card structure, and a 2-D look at
fraud vs normal.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))   # template root (src/)
BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")
SCALE = "full"                               # flip to "xl" / "xxl" to inspect those runs

In [ ]:
import pyarrow.dataset as pads

es = pads.dataset(f"{BASE}/embeddings/{SCALE}", format="parquet")
print(f"{es.count_rows():,} embeddings; columns: {es.schema.names}")
df = es.head(50_000).to_pandas()
X = np.vstack(df["embedding"].to_numpy()).astype(np.float32)
print("matrix:", X.shape)

In [ ]:
# Collapse check (same math as src/embed.embedding_health): mean pairwise cosine -> 1.0 = collapsed.
Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
s = Xn.sum(axis=0); n = len(Xn)
print(f"mean pairwise cosine: {(s @ s - n) / (n * (n - 1)):.3f}   (R2 healthy ≈ 0.27)")
print(f"mean feature variance: {X.var(axis=0).mean():.3f}")

In [ ]:
# Same card over time vs different cards: consecutive windows of one card should be far more
# similar than windows from two random cards — the embedding encodes card behavior.
card = df["card_id"].value_counts().index[0]
mine = df[df.card_id == card].sort_values("raw_ts")
M = np.vstack(mine["embedding"].to_numpy()); Mn = M / np.linalg.norm(M, axis=1, keepdims=True)
same = (Mn[:-1] * Mn[1:]).sum(1).mean()
rand = (Xn[np.random.default_rng(0).choice(n, 2000)] @ Xn[np.random.default_rng(1).choice(n, 2000)].T).mean()
print(f"card {card}: {len(mine)} windows | same-card consecutive cos {same:.3f} vs random-pair cos {rand:.3f}")

In [ ]:
# 2-D picture: PCA of a fraud-enriched sample. Frauds should clump (bursts share history).
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

fr = df[df.label == 1]; nm = df[df.label == 0].sample(min(3000, len(df)), random_state=0)
sub = pd.concat([fr, nm]); Z = PCA(2).fit_transform(np.vstack(sub["embedding"].to_numpy()))
plt.figure(figsize=(6, 5))
plt.scatter(*Z[sub.label == 0].T, s=4, alpha=0.3, label="normal")
plt.scatter(*Z[sub.label == 1].T, s=10, alpha=0.8, label="fraud")
plt.legend(); plt.title(f"{SCALE} embeddings, PCA-2 ({len(fr)} frauds in sample)");

**What to check:** mean pairwise cosine well below 0.9 (no collapse); same-card consecutive
similarity ≫ random-pair; visible (not necessarily separable — that's XGBoost's job in 512-d)
fraud structure in PCA-2. Remember PCA-2 is for eyeballs only — the headline result specifically
avoids PCA.

**Fulltest variants:** `embeddings/<scale>_fulltest` holds ~3.5M embeddings (every benchmark_fulltest row, incl. the full 2.44M-row test period) extracted with the same checkpoint — point the dataset path at it to inspect. Same health checks apply.